# Conditional SE(3) Diffusion Training (STAR-MD)

Trains a **conditional** STAR-MD score network on a single protein MD
trajectory using the SinFusion single-trajectory protocol with
Diffusion Forcing.

**What you get:** A model that autoregressively generates MD trajectories
frame-by-frame, conditioned on previous frames.

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive (persistent storage) |
| 2 | Clone repo & init submodules |
| 3 | Configure protein & training |
| 4 | Download & preprocess trajectory |
| 5 | Train (STAR-MD with Diffusion Forcing) |
| 6 | Generate trajectory (autoregressive rollout) |

## Step 1: Environment Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print('Drive mounted')

## Step 2: Clone Repository & Init Submodules

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/JiwonJJeong/winter-gen-pproject.git'

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run(['git', 'submodule', 'update', '--init', '--recursive'], check=True)
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print('data/ -> /content/drive/MyDrive/protein_data/data')
    print('Code ready')
else:
    assert os.path.isdir('gen_model'), 'Run from repo root'
    print(f'Local repo: {os.path.abspath(".")}')

## Step 3: Configuration

Edit the variables below for your protein and training preferences.

In [ ]:
# ---- Protein (single-trajectory, SinFusion-style) ----
PROTEIN   = '4o66_C'   # Protein name (as in atlas.csv)
REPLICA   = '1'        # Which replica (1, 2, or 3)

# ---- Training ----
MAX_STEPS  = 200_000   # Total training steps
BATCH_SIZE = 8
LR         = 1e-4
GRAD_CLIP  = 1.0       # Gradient clipping (MDGen default)
EMA_DECAY  = 0.999     # EMA decay (0 = disabled)

# ---- STAR-MD / Diffusion Forcing ----
NUM_FRAMES = 16        # Window length L (paper: 80; 16 for limited data)
MIN_T      = 0.01      # Min diffusion time
MAX_T      = 0.1       # Max diffusion time (narrow noise for Diffusion Forcing)
NS_PER_FRAME = 0.1     # Physical time between stored frames (ns)
CURRICULUM = True       # SinFusion-style curriculum for delta_t

# ---- Paths ----
DATA_DIR = './data'
SAVE_DIR = f'checkpoints/conditional/{PROTEIN}'

print(f'Protein    : {PROTEIN}_R{REPLICA}')
print(f'Steps      : {MAX_STEPS:,}')
print(f'Window (L) : {NUM_FRAMES}')
print(f'Curriculum : {CURRICULUM}')
print(f'Save to    : {SAVE_DIR}')

## Step 4: Download & Prepare Data

In [ ]:
import os

if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError('data/ not linked to Drive. Run Step 1 first.')

!python scripts/download_and_prep.py {PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

traj_path = f'{DATA_DIR}/{PROTEIN}/{PROTEIN}_R{REPLICA}_latent.npy'
if os.path.exists(traj_path):
    import numpy as np
    arr = np.load(traj_path)
    print(f'Data ready: {traj_path}  shape={arr.shape}')
else:
    print(f'ERROR: {traj_path} not found')

## Step 5: Train (STAR-MD + Diffusion Forcing)

Runs `gen_model/train_conditional.py` with:
- STAR-MD spatio-temporal attention (block-causal, 2D-RoPE, AdaLN)
- Diffusion Forcing: all L frames noised at same tau, loss at all positions
- SinFusion curriculum for delta_t (gradually increasing temporal range)
- EMA, gradient clipping, cosine LR, mixed precision

In [ ]:
curriculum_flag = '--curriculum' if CURRICULUM else '--no_curriculum'

!python gen_model/train_conditional.py \
    --protein {PROTEIN} \
    --replica {REPLICA} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --max_steps {MAX_STEPS} \
    --lr {LR} \
    --grad_clip {GRAD_CLIP} \
    --ema_decay {EMA_DECAY} \
    --num_frames {NUM_FRAMES} \
    --ns_per_stored_frame {NS_PER_FRAME} \
    --min_t {MIN_T} \
    --max_t {MAX_T} \
    {curriculum_flag} \
    --save_dir {SAVE_DIR} \
    --num_workers 2

## Step 6: Generate Trajectory (Autoregressive Rollout)

Generate an MD trajectory by rolling out one frame at a time
using the trained conditional model.

In [ ]:
import glob

TOTAL_FRAMES = 100     # Number of frames to generate
DELTA_T      = 0.1     # Physical stride between frames (ns)
N_STEPS      = 200     # Reverse diffusion steps per frame

ckpts = sorted(glob.glob(f'{SAVE_DIR}/cond-*.ckpt'))
best_ckpt = ckpts[-1] if ckpts else f'{SAVE_DIR}/last.ckpt'
print(f'Checkpoint: {best_ckpt}')

!python gen_model/inference_conditional.py \
    --checkpoint {best_ckpt} \
    --data_dir {DATA_DIR} \
    --protein {PROTEIN} \
    --total_frames {TOTAL_FRAMES} \
    --num_frames {NUM_FRAMES} \
    --delta_t {DELTA_T} \
    --n_steps {N_STEPS} \
    --output outputs/conditional/{PROTEIN}/traj.pt

## Step 7: Visualize Trajectory

In [ ]:
import torch
import matplotlib.pyplot as plt

traj_out = f'outputs/conditional/{PROTEIN}/traj.pt'
if os.path.exists(traj_out):
    traj = torch.load(traj_out, map_location='cpu')  # [T, N, 7]
    print(f'Trajectory: {traj.shape[0]} frames, {traj.shape[1]} residues')

    # Plot CA translation traces over time
    ca = traj[:, :, 4:7].numpy()  # translations [T, N, 3]
    rmsd_from_start = ((ca - ca[0:1]) ** 2).sum(axis=-1).mean(axis=-1) ** 0.5

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(rmsd_from_start)
    ax1.set_xlabel('Frame'); ax1.set_ylabel('RMSD from frame 0')
    ax1.set_title('Trajectory drift')

    # Overlay first and last frame
    ax2.plot(ca[0, :, 0], ca[0, :, 1], '-', lw=0.5, label='Frame 0')
    ax2.plot(ca[-1, :, 0], ca[-1, :, 1], '-', lw=0.5, label=f'Frame {len(ca)-1}')
    ax2.set_aspect('equal'); ax2.legend(); ax2.axis('off')
    ax2.set_title('First vs last frame (XY)')

    plt.suptitle(f'{PROTEIN} — Generated trajectory ({traj.shape[0]} frames)')
    plt.tight_layout(); plt.show()
else:
    print('No trajectory found. Run inference first.')